In [1]:
import numpy as np
import os
import time
from fury import window, actor

# Data and Class imports
from data import data, affine, labels, bvals, bvecs, vox_size
from env_main import UnifiedTractographyEnv
from agent import RewardDrivenAgent,ContinuousRewardAgent

In [14]:
import numpy as np

class LookAheadContinuousAgent:
    """
    An agent that evaluates directions based on 'Future Value' rather 
    than just immediate reward.
    """
    def __init__(self, env, look_ahead_steps=3, num_samples=32, gamma=0.9):
        self.env = env
        self.look_ahead_steps = look_ahead_steps # How many steps to "see" into the future
        self.num_samples = num_samples
        self.gamma = gamma # Discount factor for future rewards

    def _evaluate_path(self, start_pos, direction):
        """Simulates a path and returns the total discounted reward."""
        total_reward = 0
        current_pos = start_pos
        
        for t in range(self.look_ahead_steps):
            next_pos = current_pos + direction * self.env.step_size
            
            # Use internal reward logic
            r = self.env._compute_reward(current_pos, next_pos, direction)
            total_reward += (self.gamma ** t) * r
            
            # Update position for simulation
            current_pos = next_pos
            
        return total_reward

    def predict(self, obs):
        current_pos = obs[0:3]
        prev_dir = obs[3:6]
        
        # 1. Generate candidate directions (spherical cap)
        # We sample around the previous direction to maintain momentum
        candidates = self._generate_search_vectors(prev_dir)
        
        best_val = -np.inf
        best_dir = prev_dir if np.linalg.norm(prev_dir) > 0 else np.array([1, 0, 0])

        # 2. Look Ahead!
        for d in candidates:
            # How good is this direction over the next 3 steps?
            path_value = self._evaluate_path(current_pos, d)
            
            if path_value > best_val:
                best_val = path_value
                best_dir = d
                
        return best_dir

    def _generate_search_vectors(self, center_dir):
        # (Simplified: generates random vectors within a 45-degree cone)
        samples = []
        for _ in range(self.num_samples):
            vec = np.random.randn(3)
            vec /= np.linalg.norm(vec)
            # Ensure it points generally forward
            if np.linalg.norm(center_dir) > 0:
                if np.dot(vec, center_dir) < 0.7: # Within ~45 deg
                    continue
            samples.append(vec)
        return samples if samples else [center_dir]

    def generate_streamline(self, seed_vox):
        obs, _ = self.env.reset()
        self.env.pos = seed_vox.astype(np.float32)
        self.env.streamline = [self.env.pos.copy()]
        
        done = False
        while not done:
            best_vector = self.predict(obs)
            obs, reward, terminated, truncated, info = self.env.manual_step(best_vector)
            done = terminated or truncated
            
        return self.env.get_world_streamline()

In [2]:
env = UnifiedTractographyEnv(
    data=data, affine=affine, labels=labels, 
    bvals=bvals, bvecs=bvecs, vox_size=vox_size,
    fa_threshold=0.25, max_curvature_deg=45
)
agent = ContinuousRewardAgent(env)

In [15]:
agent = LookAheadContinuousAgent(env)

In [16]:
seed_indices = np.argwhere(labels == 2)
np.random.shuffle(seed_indices)

In [17]:
total_episodes = 10  # Run 100 episodes
all_streamlines = [[] for _ in range(total_episodes)]

print(f"Beginning Reward-Driven Tracking for {total_episodes} episodes...")

for i in range(total_episodes):
    if(i): print(f"Running {i} episode")
    # Generate one streamline
    for seed in seed_indices:
        streamline = agent.generate_streamline(seed)

        if len(streamline) > 1:
            all_streamlines[i].append(streamline)

Beginning Reward-Driven Tracking for 10 episodes...
Running 1 episode
Running 2 episode
Running 3 episode
Running 4 episode
Running 5 episode
Running 6 episode
Running 7 episode
Running 8 episode
Running 9 episode


In [22]:
scene=window.Scene()
n_st=6
env.render_streamlines(scene,streamlines=all_streamlines[n_st])
env.render_wm_surface(scene)
env.render_seeds_mask(scene)
window.show(scene,size=(800,800))

In [21]:
for i in range(len(all_streamlines)):
    print(f"length of {i} streamlines = {len(all_streamlines[i])}")

length of 0 streamlines = 438
length of 1 streamlines = 427
length of 2 streamlines = 424
length of 3 streamlines = 431
length of 4 streamlines = 428
length of 5 streamlines = 423
length of 6 streamlines = 439
length of 7 streamlines = 423
length of 8 streamlines = 430
length of 9 streamlines = 416


In [23]:
scene.clear()

In [24]:
mask=labels==3
env.render_bval_bvec(scene,mask)
window.show(scene,size=(800,800))

In [25]:
env.render_wm_surface(scene)
window.show(scene,size=(800,800))